# Wobus et al. (2021) CONUS warming-level temperature series

This notebook creates one annual CONUS-average near-surface air temperature series for every LOCA2 model, ensemble member, and scenario. It retains absolute annual temperature for 1985–2094 and calculates the 2001–2020 baseline used to interpret warming levels.

The spatial mean is area weighted using `cos(latitude)` over grid-cell centers inside the contiguous United States. Illinois is not used as the spatial domain. The model, member, and scenario dimensions remain separate, and no ensemble average is taken.

In [1]:
# # ============================================================
# # DOWNLOAD FULL-CONUS LOCA2 DAILY TEMPERATURE DATA
# # Wobus et al. (2021)
# #
# # Source structure:
# # /~pierce/LOCA2/NAmer/
# #   MODEL/0p0625deg/MEMBER/SCENARIO/VARIABLE/
# #
# # Downloads:
# #   tasmin and tasmax
# #   historical, ssp245, and ssp370
# #   daily full-CONUS files only
# #
# # Existing files are skipped.
# # ============================================================

# from pathlib import Path
# import requests
# from bs4 import BeautifulSoup
# from tqdm.auto import tqdm


# # ============================================================
# # Paths and settings
# # ============================================================

# DATA_ROOT = Path(
#     "/data/keeling/a/tahsina2/b/Climate_Impact/Data"
# )

# CONUS_INPUT_ROOT = (
#     DATA_ROOT / "LOCA2_CONUS"
# )

# CONUS_INPUT_ROOT.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# LOCA2_ROOT = (
#     "https://cirrus.ucsd.edu/~pierce/LOCA2/NAmer"
# )

# LOCA2_GRID = "0p0625deg"

# VARIABLES = (
#     "tasmin",
#     "tasmax",
# )

# SCENARIOS = (
#     "historical",
#     "ssp245",
#     "ssp370",
# )

# OVERWRITE_DOWNLOADS = False

# # Use None to download all available models.
# # Example:
# # DOWNLOAD_ONLY_MODELS = {"ACCESS-CM2", "AWI-CM-1-1-MR"}
# DOWNLOAD_ONLY_MODELS = None


# # ============================================================
# # Web directory helpers
# # ============================================================

# def get_links(url):
#     """
#     Return directory or file names listed at a LOCA2 URL.
#     """
#     response = requests.get(
#         url,
#         timeout=120,
#     )

#     if response.status_code == 404:
#         return []

#     response.raise_for_status()

#     soup = BeautifulSoup(
#         response.text,
#         "html.parser",
#     )

#     links = []

#     for anchor in soup.find_all("a"):
#         href = anchor.get("href")

#         if not href:
#             continue

#         href = href.strip("/")

#         if href in {"", ".."}:
#             continue

#         if href.startswith("?"):
#             continue

#         if "Parent Directory" in anchor.text:
#             continue

#         links.append(href)

#     return sorted(set(links))


# def list_daily_nc_files(url):
#     """
#     List daily full-CONUS NetCDF files.

#     Excludes:
#         - monthly files
#         - county-average files
#         - md5 checksum files
#     """
#     response = requests.get(
#         url,
#         timeout=120,
#     )

#     # Some members or scenarios are unavailable.
#     if response.status_code == 404:
#         return []

#     response.raise_for_status()

#     soup = BeautifulSoup(
#         response.text,
#         "html.parser",
#     )

#     files = []

#     for anchor in soup.find_all("a"):
#         href = anchor.get("href")

#         if not href:
#             continue

#         filename = href.strip("/")

#         if not filename.endswith(".nc"):
#             continue

#         if filename.endswith(".monthly.nc"):
#             continue

#         if ".monthly." in filename:
#             continue

#         if filename.endswith(".US_county_avg.nc"):
#             continue

#         files.append(
#             f"{url.rstrip('/')}/{filename}"
#         )

#     return sorted(set(files))


# # ============================================================
# # Discover models, members, and files
# # ============================================================

# models = [
#     model
#     for model in get_links(
#         f"{LOCA2_ROOT}/"
#     )
#     if not model.endswith(".txt")
# ]

# if DOWNLOAD_ONLY_MODELS is not None:
#     models = [
#         model
#         for model in models
#         if model in DOWNLOAD_ONLY_MODELS
#     ]

# print(f"Models selected: {len(models)}")

# tasks = []

# for model in models:

#     model_grid_url = (
#         f"{LOCA2_ROOT}/"
#         f"{model}/"
#         f"{LOCA2_GRID}/"
#     )

#     members = get_links(model_grid_url)

#     for member in members:

#         if member.endswith(".txt"):
#             continue

#         for scenario in SCENARIOS:

#             for variable in VARIABLES:

#                 directory_url = (
#                     f"{model_grid_url}"
#                     f"{member}/"
#                     f"{scenario}/"
#                     f"{variable}/"
#                 )

#                 files = list_daily_nc_files(
#                     directory_url
#                 )

#                 for file_url in files:

#                     filename = file_url.rsplit(
#                         "/",
#                         1,
#                     )[-1]

#                     tasks.append(
#                         {
#                             "model": model,
#                             "member": member,
#                             "scenario": scenario,
#                             "variable": variable,
#                             "url": file_url,
#                             "filename": filename,
#                         }
#                     )

# print(f"Daily files discovered: {len(tasks)}")

# if not tasks:
#     raise RuntimeError(
#         "No daily LOCA2 files were found. "
#         "Check LOCA2_ROOT and the selected models."
#     )


# # ============================================================
# # Download helper
# # ============================================================

# def download_file(
#     url,
#     output_file,
# ):
#     output_file = Path(output_file)

#     output_file.parent.mkdir(
#         parents=True,
#         exist_ok=True,
#     )

#     if (
#         output_file.exists()
#         and output_file.stat().st_size > 0
#         and not OVERWRITE_DOWNLOADS
#     ):
#         return "skipped"

#     temporary_file = output_file.with_suffix(
#         output_file.suffix + ".part"
#     )

#     with requests.get(
#         url,
#         stream=True,
#         timeout=600,
#     ) as response:

#         response.raise_for_status()

#         with temporary_file.open("wb") as handle:
#             for chunk in response.iter_content(
#                 chunk_size=8 * 1024 * 1024
#             ):
#                 if chunk:
#                     handle.write(chunk)

#     temporary_file.replace(output_file)

#     return "downloaded"


# # ============================================================
# # Download full-CONUS files
# # ============================================================

# downloaded = 0
# skipped = 0
# failed = 0

# for task in tqdm(
#     tasks,
#     desc="Downloading LOCA2 full-CONUS temperature data",
#     unit="file",
# ):

#     output_file = (
#         CONUS_INPUT_ROOT
#         / task["variable"]
#         / task["scenario"]
#         / task["model"]
#         / task["member"]
#         / task["filename"]
#     )

#     try:

#         status = download_file(
#             task["url"],
#             output_file,
#         )

#         if status == "downloaded":
#             downloaded += 1
#         else:
#             skipped += 1

#     except Exception as error:

#         failed += 1

#         print(
#             f"FAILED: "
#             f"{task['model']}/"
#             f"{task['member']}/"
#             f"{task['scenario']}/"
#             f"{task['variable']}/"
#             f"{task['filename']}\n"
#             f"{type(error).__name__}: {error}"
#         )

# print()
# print("Download complete")
# print("Downloaded:", downloaded)
# print("Skipped:", skipped)
# print("Failed:", failed)
# print("Saved under:", CONUS_INPUT_ROOT)


# # ============================================================
# # Verify one downloaded file
# # ============================================================

# downloaded_files = sorted(
#     CONUS_INPUT_ROOT.rglob("*.nc")
# )

# print(
#     f"Total downloaded NetCDF files: "
#     f"{len(downloaded_files)}"
# )

# if downloaded_files:
#     print(
#         "Example file:",
#         downloaded_files[0],
#     )

In [2]:
# Imports and paths
from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import regionmask
from tqdm.auto import tqdm

# The input root must contain full-CONUS or CONUS-covering LOCA2 data.
# The Illinois-cropped directory is not sufficient for a CONUS mean.
DATA_ROOT = Path("/data/keeling/a/tahsina2/b/Climate_Impact/Data")
CONUS_INPUT_ROOT = DATA_ROOT / "LOCA2_CONUS"
OUTPUT_ROOT = DATA_ROOT / "LOCA2_Illinois" / "Wobus et al. (2021)"

# Use an existing contiguous-US boundary shapefile if available.
# Set this to the path on Keeling before running.
CONUS_SHAPEFILE = DATA_ROOT / "CONUS_Boundaries" / "cb_2018_us_nation_5m.shp"

SCENARIOS = ("ssp245", "ssp370")
PERIODS = {
    "historical_1985_2014": (1985, 2014),
    "near_2015_2044": (2015, 2044),
    "mid_2035_2064": (2035, 2064),
    "late_2065_2094": (2065, 2094),
}
BASELINE_START = 2001
BASELINE_END = 2020
OVERWRITE = False
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Input root:", CONUS_INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Boundary file:", CONUS_SHAPEFILE)

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_CONUS
Output root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)
Boundary file: /data/keeling/a/tahsina2/b/Climate_Impact/Data/CONUS_Boundaries/cb_2018_us_nation_5m.shp


## Input layout

The reader supports either a direct `tas` variable or LOCA2 `tasmin` and `tasmax`. Expected folders are:

```text
<CONUS_INPUT_ROOT>/tas/<scenario>/<model>/<member>/*.nc
```

or, when daily mean temperature must be calculated:

```text
<CONUS_INPUT_ROOT>/tasmin/<scenario>/<model>/<member>/*.nc
<CONUS_INPUT_ROOT>/tasmax/<scenario>/<model>/<member>/*.nc
```

Historical data are read from `historical/<model>/<member>/`; future data are read from the selected SSP folder. The notebook keeps each model-member-scenario series separate.

In [12]:
def nc_files(path):
    return sorted(Path(path).glob("*.nc"))

def safe_token(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))

def find_variable(ds, requested):
    aliases = {"tas": ["tas", "tmean", "air_temperature"]}
    for name in aliases.get(requested, [requested]):
        if name in ds.data_vars:
            return name
    raise KeyError(f"Could not find {requested}; available variables: {list(ds.data_vars)}")

def open_files(files, requested="tas"):
    if not files:
        raise FileNotFoundError("No NetCDF files found")
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        join="outer",
        data_vars="all",
        combine_attrs="override",
    )
    name = find_variable(ds, requested)
    data = ds[name].sortby("time")
    _, keep = np.unique(data.time.values, return_index=True)
    return data.isel(time=np.sort(keep))

def historical_dir(variable, model, member):
    exact = CONUS_INPUT_ROOT / variable / "historical" / model / member
    if nc_files(exact):
        return exact
    model_root = CONUS_INPUT_ROOT / variable / "historical" / model
    candidates = [p for p in model_root.glob("*/") if nc_files(p)] if model_root.is_dir() else []
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(f"Cannot identify historical {variable} for {model}/{member}")

def combine_series(variable, model, member, scenario):
    hist = open_files(nc_files(historical_dir(variable, model, member)), variable)
    future = open_files(nc_files(CONUS_INPUT_ROOT / variable / scenario / model / member), variable)
    hist = hist.sel(time=(hist.time.dt.year >= 1985) & (hist.time.dt.year <= 2014))
    future = future.sel(time=(future.time.dt.year >= 2015) & (future.time.dt.year <= 2094))
    combined = xr.concat([hist, future], dim="time").sortby("time")
    _, keep = np.unique(combined.time.values, return_index=True)
    return combined.isel(time=np.sort(keep))

def discover_runs(variable="tas"):
    jobs=[]
    root=CONUS_INPUT_ROOT / variable
    for scenario in SCENARIOS:
        ref=root/scenario
        if not ref.is_dir(): continue
        for model_dir in sorted(p for p in ref.iterdir() if p.is_dir()):
            for member_dir in sorted(p for p in model_dir.iterdir() if p.is_dir()):
                if nc_files(member_dir): jobs.append((model_dir.name,member_dir.name,scenario))
    return jobs

In [13]:
from pathlib import Path
import geopandas as gpd
import regionmask

CONUS_SHAPEFILE = Path(
    "/data/keeling/a/tahsina2/d/Climate_Impact/Data/"
    "CONUS_Boundaries/cb_2023_us_state_500k.zip"
)

if not CONUS_SHAPEFILE.exists():
    raise FileNotFoundError(
        f"Boundary ZIP not found: {CONUS_SHAPEFILE}"
    )

boundaries = gpd.read_file(
    f"zip://{CONUS_SHAPEFILE.resolve()}"
).to_crs("EPSG:4326")

excluded = {"AK", "HI", "PR", "VI", "GU", "AS", "MP"}

boundaries = boundaries.loc[
    ~boundaries["STUSPS"].isin(excluded)
].copy()

if len(boundaries) != 49:
    raise ValueError(
        "Expected 48 contiguous states plus Washington, DC; "
        f"found {len(boundaries)} features."
    )

conus_geometry = (
    boundaries.geometry.union_all()
    if hasattr(boundaries.geometry, "union_all")
    else boundaries.geometry.unary_union
)

conus_regions = regionmask.Regions(
    outlines=[conus_geometry],
    numbers=[0],
    names=["CONUS"],
    abbrevs=["CONUS"],
)

print("Boundary features:", len(boundaries))
print("CONUS region mask definition is ready.")

Boundary features: 49
CONUS region mask definition is ready.


In [14]:
def daily_mean_from_minmax(model, member, scenario):
    tmin = combine_series("tasmin", model, member, scenario)
    tmax = combine_series("tasmax", model, member, scenario)
    tmin, tmax = xr.align(tmin, tmax, join="inner")
    if tmin.sizes.get("time", 0) == 0:
        raise ValueError("No shared dates between tasmin and tasmax")
    return ((tmin + tmax) / 2).rename("tas")

def load_temperature(model, member, scenario):
    tas_root = CONUS_INPUT_ROOT / "tas"
    if nc_files(tas_root / scenario / model / member):
        return combine_series("tas", model, member, scenario)
    return daily_mean_from_minmax(model, member, scenario)

def aggregate_conus_annual(tas):
    # LOCA2 dimensions are normally lat/lon. The code also accepts latitude/longitude.
    lat_name = "lat" if "lat" in tas.coords else "latitude"
    lon_name = "lon" if "lon" in tas.coords else "longitude"
    tas = tas.rename({lat_name:"lat", lon_name:"lon"}).sortby("lat").sortby("lon")
    mask = conus_regions.mask(tas.lon, tas.lat)
    inside = mask == 0
    area_weight = xr.DataArray(np.cos(np.deg2rad(tas.lat)), dims=("lat",), coords={"lat":tas.lat})
    masked = tas.where(inside)
    weights = area_weight.broadcast_like(masked)
    # Area-weighted CONUS mean for each month.
    # Latitude must be in degrees.
    area_weights = np.cos(np.deg2rad(masked["lat"]))
    
    monthly_conus = masked.weighted(area_weights).mean(
        dim=("lat", "lon"),
        skipna=True,
    )
    
    # Require consecutive monthly records.
    month_index = (
        monthly_conus.time.dt.year.values * 12
        + monthly_conus.time.dt.month.values
    )
    
    if len(month_index) == 0:
        raise ValueError("No monthly temperature records found.")
    
    if np.any(np.diff(month_index) != 1):
        raise ValueError(
            "Monthly temperature records contain gaps, duplicates, "
            "or are not chronologically ordered."
        )
    
    # Weight monthly means by month length using the source calendar.
    days = monthly_conus.time.dt.days_in_month.astype("float64")
    
    annual = (
        (monthly_conus * days)
        .groupby("time.year")
        .sum(dim="time", skipna=False)
        / days.groupby("time.year").sum(dim="time")
    )
    
    # Reject incomplete years instead of returning partial-year means.
    months_per_year = xr.ones_like(days).groupby("time.year").sum(
        dim="time"
    )
    
    incomplete_years = months_per_year.year.where(
        months_per_year != 12,
        drop=True,
    ).values
    
    if len(incomplete_years):
        raise ValueError(
            f"Incomplete years in monthly input: {incomplete_years.tolist()}"
        )
    
    annual.attrs = dict(masked.attrs)
    annual.attrs["aggregation"] = (
        "cosine-latitude weighted CONUS monthly mean, "
        "then month-length weighted annual mean"
    )
    annual = annual.rename({"year":"calendar_year"})
    annual.attrs.update(units="degree_Celsius", long_name="CONUS area-weighted annual mean near-surface air temperature", spatial_domain="contiguous United States", spatial_weighting="cosine latitude grid-cell area weighting")
    return annual

def create_result(model, member, scenario):
    daily = load_temperature(model, member, scenario)
    annual = aggregate_conus_annual(daily)
    baseline = annual.sel(calendar_year=slice(BASELINE_START, BASELINE_END)).mean("calendar_year", skipna=False)
    warming = annual - baseline
    result = xr.Dataset({"conus_annual_mean_temperature":annual, "conus_warming_relative_to_2001_2020":warming, "baseline_2001_2020_temperature":baseline})
    result.attrs.update(paper="Wobus et al. (2021)", model=model, member=member, scenario=scenario, baseline_period="2001-2020", absolute_temperature_units="degrees Celsius", spatial_domain="contiguous United States", spatial_weighting="cosine latitude grid-cell area weighting", ensemble_handling="one series per model-member-scenario; no ensemble average", scenarios="ssp245 and ssp370")
    return result

In [15]:
from pathlib import Path

CONUS_INPUT_ROOT = Path(
    "/data/keeling/a/tahsina2/d/Climate_Impact/Data/LOCA2_CONUS"
)

if not CONUS_INPUT_ROOT.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {CONUS_INPUT_ROOT}"
    )

for variable in ("tasmin", "tasmax"):
    for scenario in ("historical", "ssp245", "ssp370"):
        folder = CONUS_INPUT_ROOT / variable / scenario
        count = sum(
            1 for _ in folder.rglob("*.monthly.CONUS.nc")
        )
        print(f"{variable}/{scenario}: {count} completed files")

jobs = discover_runs("tasmin")

print(f"\nEligible future runs: {len(jobs)}")
print(jobs[:5])

if not jobs:
    raise FileNotFoundError(
        "No eligible runs found. If completed file counts are positive, "
        "check discover_runs() for its input path and filename pattern."
    )

tasmin/historical: 108 completed files
tasmin/ssp245: 192 completed files
tasmin/ssp370: 276 completed files
tasmax/historical: 108 completed files
tasmax/ssp245: 192 completed files
tasmax/ssp370: 276 completed files

Eligible future runs: 156
[('ACCESS-CM2', 'r1i1p1f1', 'ssp245'), ('ACCESS-CM2', 'r2i1p1f1', 'ssp245'), ('ACCESS-CM2', 'r3i1p1f1', 'ssp245'), ('ACCESS-ESM1-5', 'r1i1p1f1', 'ssp245'), ('ACCESS-ESM1-5', 'r2i1p1f1', 'ssp245')]


In [16]:
# Write one file per model-member-scenario
for model, member, scenario in tqdm(jobs, desc="Wobus et al. (2021)", unit="run"):
    outfile = OUTPUT_ROOT / f"Wobus_et_al_2021__{safe_token(model)}_{safe_token(member)}_{scenario}.nc"
    if outfile.exists() and not OVERWRITE:
        print("SKIPPED existing:", outfile.name)
        continue
    try:
        result = create_result(model, member, scenario)
        result.to_netcdf(outfile, encoding={v:{"zlib":True,"complevel":4,"shuffle":True} for v in result.data_vars})
        result.close()
        print("WROTE:", outfile)
    except Exception as exc:
        print(f"FAILED {model}/{member}/{scenario}: {type(exc).__name__}: {exc}")

Wobus et al. (2021):   0%|          | 0/156 [00:00<?, ?run/s]

WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-CM2_r1i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-CM2_r2i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-CM2_r3i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-ESM1-5_r1i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-ESM1-5_r2i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-ESM1-5_r3i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Wobus et al. (2021)/Wobus_et_al_2021__ACCESS-ESM1-5_r4i1p1f1_ssp245.nc
WROTE: /data/keeling/a/tahsin

In [17]:
# Inspect a generated file
files = sorted(OUTPUT_ROOT.glob("Wobus_et_al_2021__*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as ds:
        display(ds)
else:
    print("No output files found.")

<xarray.Dataset> Size: 3kB
Dimensions:                              (calendar_year: 110)
Coordinates:
  * calendar_year                        (calendar_year) int64 880B 1985 ... ...
Data variables:
    conus_annual_mean_temperature        (calendar_year) float64 880B ...
    conus_warming_relative_to_2001_2020  (calendar_year) float64 880B ...
    baseline_2001_2020_temperature       float64 8B ...
Attributes:
    paper:                       Wobus et al. (2021)
    model:                       ACCESS-CM2
    member:                      r1i1p1f1
    scenario:                    ssp245
    baseline_period:             2001-2020
    absolute_temperature_units:  degrees Celsius
    spatial_domain:              contiguous United States
    spatial_weighting:           cosine latitude grid-cell area weighting
    ensemble_handling:           one series per model-member-scenario; no ens...
    scenarios:                   ssp245 and ssp370